In [86]:
import numpy as np
import csv
from collections import Counter

# Import sklearn for comparison
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize
from random import seed, sample

class TreeConfig:
    """Configuration for decision tree"""

    def __init__(self, random_seed=42, test_size=0.3, 
                 max_depth=None, min_samples_split=2, min_samples_leaf=1, 
                 criterion='gini', min_impurity_decrease=0.0,
                 max_features=None, splitter='best'):
        self.random_seed = random_seed
        self.test_size = test_size
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion  # 'gini' or 'entropy'
        self.min_impurity_decrease = min_impurity_decrease
        self.max_features = max_features
        self.splitter = splitter  # 'best' or 'random'
        seed(self.random_seed) if self.random_seed else None

# IMPROVED SPLIT CRITERIA
def gini_index(groups, classes):
    """Calculate Gini impurity (vectorized)"""
    n_instances = sum(len(g) for g in groups)
    if n_instances == 0:
        return 0

    gini = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue

        labels = np.array([row[-1] for row in group])
        class_counts = np.array([np.sum(labels == c) for c in classes])
        probabilities = class_counts / size
        score = 1.0 - np.sum(probabilities ** 2)
        gini += score * (size / n_instances)

    return gini

def entropy_index(groups, classes):
    """Calculate entropy (information gain)"""
    n_instances = sum(len(g) for g in groups)
    if n_instances == 0:
        return 0

    entropy = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue

        labels = np.array([row[-1] for row in group])
        class_counts = np.array([np.sum(labels == c) for c in classes])
        probabilities = class_counts / size

        # Avoid log(0)
        probabilities = probabilities[probabilities > 0]
        score = -np.sum(probabilities * np.log2(probabilities))
        entropy += score * (size / n_instances)

    return entropy

def calculate_impurity(groups, classes, criterion='gini'):
    """Calculate impurity based on criterion"""
    if criterion == 'entropy':
        return entropy_index(groups, classes)
    else:
        return gini_index(groups, classes)

# OPTIMIZED SPLITTING

def get_split_improved(dataset, config):
    """Improved split finding with vectorization"""
    if not dataset or not dataset[0]:
        return None

    class_values = list(set(row[-1] for row in dataset))
    n_features = len(dataset[0]) - 1

    # Feature selection
    if config.max_features == 'sqrt':
        n_features_to_use = max(1, int(np.sqrt(n_features)))
        feature_indices = sample(range(n_features), n_features_to_use)
    elif config.max_features == 'log2':
        n_features_to_use = max(1, int(np.log2(n_features)))
        feature_indices = sample(range(n_features), n_features_to_use)
    elif isinstance(config.max_features, int):
        n_features_to_use = min(config.max_features, n_features)
        feature_indices = sample(range(n_features), n_features_to_use)
    else:
        feature_indices = range(n_features)

    best = {'index': 0, 'value': 0, 'score': float('inf'), 'groups': None, 
            'n_samples': len(dataset)}

    # Vectorize dataset for faster operations
    dataset_array = np.array(dataset)

    for i in feature_indices:
        feature_values = dataset_array[:, i]
        unique_values = np.unique(feature_values)

        if len(unique_values) < 2:
            continue

        # Efficient split candidates (max 20 per feature)
        if len(unique_values) > 20:
            split_candidates = np.percentile(unique_values, np.linspace(5, 95, 19))
        else:
            split_candidates = (unique_values[:-1] + unique_values[1:]) / 2

        for split_val in split_candidates:
            left_mask = feature_values < split_val

            # Quick validation
            left_size = np.sum(left_mask)
            right_size = len(feature_values) - left_size

            if (left_size == 0 or right_size == 0 or 
                left_size < config.min_samples_leaf or 
                right_size < config.min_samples_leaf):
                continue

            left = dataset_array[left_mask].tolist()
            right = dataset_array[~left_mask].tolist()

            groups = (left, right)
            score = calculate_impurity(groups, class_values, config.criterion)

            if score < best['score']:
                best = {
                    'index': i, 
                    'value': float(split_val), 
                    'score': score, 
                    'groups': groups,
                    'n_samples': len(dataset)
                }

    return best if best['groups'] is not None else None

# TREE BUILDING

def to_terminal(group):
    """Create leaf node with full information"""
    if not group:
        return {
            'type': 'leaf',
            'value': None,
            'samples': 0,
            'distribution': {},
            'impurity': 0.0
        }

    outcomes = [row[-1] for row in group]
    distribution = Counter(outcomes)
    majority = max(distribution, key=distribution.get)

    # Calculate leaf impurity
    total = len(outcomes)
    impurity = 1.0 - sum((count/total)**2 for count in distribution.values())

    return {
        'type': 'leaf',
        'value': majority,
        'samples': len(group),
        'distribution': dict(distribution),
        'impurity': impurity
    }

def split_node_improved(node, config, depth, all_classes):
    """Improved recursive splitting"""
    if not isinstance(node, dict) or 'groups' not in node:
        return

    left, right = node['groups']
    del node['groups']

    # Handle empty children
    if not left or not right:
        combined = left + right
        terminal = to_terminal(combined)
        node.update(terminal)
        node['left'] = node['right'] = None
        return

    # Check max depth
    if config.max_depth and depth >= config.max_depth:
        node['left'] = to_terminal(left)
        node['right'] = to_terminal(right)
        return

    # Check minimum samples
    if len(left) < config.min_samples_split and len(right) < config.min_samples_split:
        node['left'] = to_terminal(left)
        node['right'] = to_terminal(right)
        return

    # Impurity decrease check
    if config.min_impurity_decrease > 0:
        combined = left + right
        parent_impurity = calculate_impurity([combined], all_classes, config.criterion)
        child_impurity = calculate_impurity([left, right], all_classes, config.criterion)
        impurity_decrease = parent_impurity - child_impurity

        if impurity_decrease < config.min_impurity_decrease:
            node['left'] = to_terminal(left)
            node['right'] = to_terminal(right)
            return

    # Process left child
    if len(left) < config.min_samples_split:
        node['left'] = to_terminal(left)
    else:
        node['left'] = get_split_improved(left, config)
        if node['left'] is not None:
            split_node_improved(node['left'], config, depth + 1, all_classes)
        else:
            node['left'] = to_terminal(left)

    # Process right child
    if len(right) < config.min_samples_split:
        node['right'] = to_terminal(right)
    else:
        node['right'] = get_split_improved(right, config)
        if node['right'] is not None:
            split_node_improved(node['right'], config, depth + 1, all_classes)
        else:
            node['right'] = to_terminal(right)


# IMPROVED DECISION TREE CLASS


class ImprovedDecisionTree:
    """Improved Decision Tree Classifier"""

    def __init__(self, config):
        self.config = config
        self.tree = None
        self.classes = None
        self.n_features = None
        self.feature_importances_ = None
        self.max_depth_reached = 0
        self.n_leaves = 0
        self.n_nodes = 0

    def fit(self, X, y):
        """Train the decision tree"""
        # Convert to internal format
        dataset = [list(X[i]) + [y[i]] for i in range(len(X))]

        self.classes = sorted(set(y))
        self.n_features = X.shape[1] if hasattr(X, 'shape') else len(X[0])

        # Build tree
        self.tree = get_split_improved(dataset, self.config)
        if self.tree is not None:
            split_node_improved(self.tree, self.config, 1, self.classes)
        else:
            self.tree = to_terminal(dataset)

        # Calculate statistics
        self._calculate_tree_stats()
        self._calculate_feature_importances()

        return self

    def predict(self, X):
        """Predict class labels"""
        return [self._predict_single(row) for row in X]

    def _predict_single(self, row):
        """Predict single instance"""
        node = self.tree

        while isinstance(node, dict):
            if node.get('type') == 'leaf':
                return node.get('value')

            if 'index' in node and len(row) > node['index']:
                node = node['left'] if row[node['index']] < node['value'] else node['right']
            else:
                return node.get('value')

        return node

    def predict_proba(self, X):
        """Predict class probabilities"""
        probas = []
        for row in X:
            leaf = self._get_leaf(row)
            if isinstance(leaf, dict) and 'distribution' in leaf:
                total = leaf['samples']
                if total > 0:
                    probs = [leaf['distribution'].get(c, 0) / total for c in self.classes]
                else:
                    probs = [1.0 / len(self.classes)] * len(self.classes)
            else:
                pred = self._predict_single(row)
                probs = [1.0 if c == pred else 0.0 for c in self.classes]
            probas.append(probs)
        return np.array(probas)

    def _get_leaf(self, row):
        """Navigate to leaf node"""
        node = self.tree

        while isinstance(node, dict) and 'index' in node:
            if len(row) > node['index']:
                node = node['left'] if row[node['index']] < node['value'] else node['right']
            else:
                break

        return node

    def _calculate_tree_stats(self):
        """Calculate tree statistics"""
        def traverse(node, depth=0):
            if not isinstance(node, dict):
                return depth, 1, 1

            if node.get('type') == 'leaf':
                self.n_leaves += 1
                return depth, 1, 1

            self.n_nodes += 1

            left_depth, left_nodes, left_leaves = 0, 0, 0
            right_depth, right_nodes, right_leaves = 0, 0, 0

            if node.get('left'):
                left_depth, left_nodes, left_leaves = traverse(node['left'], depth + 1)
            if node.get('right'):
                right_depth, right_nodes, right_leaves = traverse(node['right'], depth + 1)

            max_depth = max(left_depth, right_depth)
            return max_depth, 1 + left_nodes + right_nodes, left_leaves + right_leaves

        self.n_leaves = 0
        self.n_nodes = 0
        self.max_depth_reached, _, _ = traverse(self.tree)

    def _calculate_feature_importances(self):
        """Calculate feature importance"""
        importance = np.zeros(self.n_features)

        def traverse(node):
            if not isinstance(node, dict) or node.get('type') == 'leaf':
                return

            if 'index' in node:
                feature_idx = node['index']
                n_samples = node.get('n_samples', 0)
                importance[feature_idx] += n_samples

                if node.get('left'):
                    traverse(node['left'])
                if node.get('right'):
                    traverse(node['right'])

        traverse(self.tree)

        if np.sum(importance) > 0:
            self.feature_importances_ = importance / np.sum(importance)
        else:
            self.feature_importances_ = importance

    def get_depth(self):
        """Get tree depth"""
        return self.max_depth_reached

    def get_n_leaves(self):
        """Get number of leaves"""
        return self.n_leaves

# EVALUATION


def calculate_metrics(y_true, y_pred, y_proba, classes):
    """Calculate comprehensive metrics"""
    accuracy = accuracy_score(y_true, y_pred)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    y_true_bin = label_binarize(y_true, classes=classes)

    if y_proba is not None and len(classes) > 1:
        try:
            if len(classes) == 2:
                fpr, tpr, _ = roc_curve(y_true_bin, y_proba[:, 1])
                auroc = auc(fpr, tpr)
            else:
                auc_scores = []
                for i in range(len(classes)):
                    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
                    auc_scores.append(auc(fpr, tpr))
                auroc = np.mean(auc_scores)
        except:
            auroc = 0.0
    else:
        auroc = 0.0

    return {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'auroc': auroc
    }

# COMPARISON RUNNER

def print_comparison_table(sk_metrics,
                          custom_metrics):
    """Print side-by-side comparison table"""
    print("\n┌──────────────┬─────────────┬─────────────┬")
    print("│ Metric       │ sklearn     │ Custom      │")
    print("├──────────────┼─────────────┼─────────────┼")
    print(f"│ Accuracy     │ {sk_metrics['accuracy']*100:>6.2f}%     │ {custom_metrics['accuracy']*100:>6.2f}%     │")
    print(f"│ F1-Score     │ {sk_metrics['f1_weighted']:>7.4f}     │ {custom_metrics['f1_weighted']:>7.4f}     │")
    print(f"│ AUROC        │ {sk_metrics['auroc']:>7.4f}     │ {custom_metrics['auroc']:>7.4f}     │")


def run_comparison(dataset_loader, dataset_name,config_name, config):
    """Compare improved custom tree with sklearn"""

    print("=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    # Load data
    data = dataset_loader()
    X, y = data.data, data.target
    classes = sorted(set(y))

    # print(f"\n Dataset Info:")
    # print(f"   Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(classes)}")

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.test_size, 
        random_state=config.random_seed, stratify=y
    )

    # sklearn Decision Tree

    print("\n sklearn DecisionTreeClassifier")
    dt_sklearn = DecisionTreeClassifier(
        max_depth=config.max_depth,
        min_samples_split=config.min_samples_split,
        min_samples_leaf=config.min_samples_leaf,
        criterion=config.criterion,
        min_impurity_decrease=config.min_impurity_decrease,
        random_state=config.random_seed
    )
    dt_sklearn.fit(X_train, y_train)
    y_pred_sk = dt_sklearn.predict(X_test)
    y_proba_sk = dt_sklearn.predict_proba(X_test)

    metrics_sk = calculate_metrics(y_test, y_pred_sk, y_proba_sk, classes)
    print("\n Custom DecisionTree")
    dt_custom = ImprovedDecisionTree(config)
    dt_custom.fit(X_train, y_train)
    y_pred_custom = dt_custom.predict(X_test)
    y_proba_custom = dt_custom.predict_proba(X_test)

    metrics_custom = calculate_metrics(y_test, y_pred_custom, y_proba_custom, classes)

    print_comparison_table(
    metrics_sk,
    metrics_custom
    )


    return {
    'dataset': dataset_name,
    'config': config_name,
    'sklearn_accuracy': metrics_sk['accuracy'],
    'sklearn_f1': metrics_sk['f1_weighted'],
    'sklearn_auroc': metrics_sk['auroc'],
    'custom_accuracy': metrics_custom['accuracy'],
    'custom_f1': metrics_custom['f1_weighted'],
    'custom_auroc': metrics_custom['auroc']
    }

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print(" " * 21 + "DECISION TREE ANALYSIS" + " " * 30 )

    # Test different configurations
    configs = [
        ("Default (Gini)", TreeConfig(max_depth=None, criterion='gini')),
        ("Depth=10 (Gini)", TreeConfig(max_depth=10, criterion='gini')),
        ("Depth=10 (Entropy)", TreeConfig(max_depth=10, criterion='entropy')),
        ("Pruned", TreeConfig(max_depth=10, min_impurity_decrease=0.01)),
    ]

    datasets = [
        (load_iris, "IRIS"),
        (load_wine, "WINE")
    ]
    results=[]
    for dataset_loader, dataset_name in datasets:
        for config_name, config in configs:
            print(f"\n\n{'='*80}")
            print(f"Configuration: {config_name}")
            print(f"{'='*80}")
            result = run_comparison(dataset_loader, dataset_name, config_name, config)
            results.append(result)

    csv_file = 'decision_tree_results.csv'
    fieldnames = ['dataset', 'config', 'sklearn_accuracy', 'sklearn_f1', 'sklearn_auroc',
                  'custom_accuracy', 'custom_f1', 'custom_auroc']
    
    with open(csv_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    print(f'\n\nResults saved to {csv_file}')         

                     DECISION TREE ANALYSIS                              


Configuration: Default (Gini)
DATASET: IRIS

 sklearn DecisionTreeClassifier

 Custom DecisionTree

┌──────────────┬─────────────┬─────────────┬
│ Metric       │ sklearn     │ Custom      │
├──────────────┼─────────────┼─────────────┼
│ Accuracy     │  93.33%     │  91.11%     │
│ F1-Score     │  0.9327     │  0.9111     │
│ AUROC        │  0.9500     │  0.9333     │


Configuration: Depth=10 (Gini)
DATASET: IRIS

 sklearn DecisionTreeClassifier

 Custom DecisionTree

┌──────────────┬─────────────┬─────────────┬
│ Metric       │ sklearn     │ Custom      │
├──────────────┼─────────────┼─────────────┼
│ Accuracy     │  93.33%     │  91.11%     │
│ F1-Score     │  0.9327     │  0.9111     │
│ AUROC        │  0.9500     │  0.9333     │


Configuration: Depth=10 (Entropy)
DATASET: IRIS

 sklearn DecisionTreeClassifier

 Custom DecisionTree

┌──────────────┬─────────────┬─────────────┬
│ Metric       │ sklearn     │

In [88]:
import numpy as np
import csv
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

# Import sklearn for comparison
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize
from random import seed, sample

# ============================================================================
# DECISION TREE CLASSES AND FUNCTIONS
# ============================================================================

class TreeConfig:
    """Configuration for decision tree"""

    def __init__(self, random_seed=42, test_size=0.3, 
                 max_depth=None, min_samples_split=2, min_samples_leaf=1, 
                 criterion='gini', min_impurity_decrease=0.0,
                 max_features=None, splitter='best'):
        self.random_seed = random_seed
        self.test_size = test_size
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion  # 'gini' or 'entropy'
        self.min_impurity_decrease = min_impurity_decrease
        self.max_features = max_features
        self.splitter = splitter  # 'best' or 'random'
        seed(self.random_seed) if self.random_seed else None

# IMPROVED SPLIT CRITERIA
def gini_index(groups, classes):
    """Calculate Gini impurity (vectorized)"""
    n_instances = sum(len(g) for g in groups)
    if n_instances == 0:
        return 0

    gini = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue

        labels = np.array([row[-1] for row in group])
        class_counts = np.array([np.sum(labels == c) for c in classes])
        probabilities = class_counts / size
        score = 1.0 - np.sum(probabilities ** 2)
        gini += score * (size / n_instances)

    return gini

def entropy_index(groups, classes):
    """Calculate entropy (information gain)"""
    n_instances = sum(len(g) for g in groups)
    if n_instances == 0:
        return 0

    entropy = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue

        labels = np.array([row[-1] for row in group])
        class_counts = np.array([np.sum(labels == c) for c in classes])
        probabilities = class_counts / size

        # Avoid log(0)
        probabilities = probabilities[probabilities > 0]
        score = -np.sum(probabilities * np.log2(probabilities))
        entropy += score * (size / n_instances)

    return entropy

def calculate_impurity(groups, classes, criterion='gini'):
    """Calculate impurity based on criterion"""
    if criterion == 'entropy':
        return entropy_index(groups, classes)
    else:
        return gini_index(groups, classes)

# OPTIMIZED SPLITTING

def get_split_improved(dataset, config):
    """Improved split finding with vectorization"""
    if not dataset or not dataset[0]:
        return None

    class_values = list(set(row[-1] for row in dataset))
    n_features = len(dataset[0]) - 1

    # Feature selection
    if config.max_features == 'sqrt':
        n_features_to_use = max(1, int(np.sqrt(n_features)))
        feature_indices = sample(range(n_features), n_features_to_use)
    elif config.max_features == 'log2':
        n_features_to_use = max(1, int(np.log2(n_features)))
        feature_indices = sample(range(n_features), n_features_to_use)
    elif isinstance(config.max_features, int):
        n_features_to_use = min(config.max_features, n_features)
        feature_indices = sample(range(n_features), n_features_to_use)
    else:
        feature_indices = range(n_features)

    best = {'index': 0, 'value': 0, 'score': float('inf'), 'groups': None, 
            'n_samples': len(dataset)}

    # Vectorize dataset for faster operations
    dataset_array = np.array(dataset)

    for i in feature_indices:
        feature_values = dataset_array[:, i]
        unique_values = np.unique(feature_values)

        if len(unique_values) < 2:
            continue

        # Efficient split candidates (max 20 per feature)
        if len(unique_values) > 20:
            split_candidates = np.percentile(unique_values, np.linspace(5, 95, 19))
        else:
            split_candidates = (unique_values[:-1] + unique_values[1:]) / 2

        for split_val in split_candidates:
            left_mask = feature_values < split_val

            # Quick validation
            left_size = np.sum(left_mask)
            right_size = len(feature_values) - left_size

            if (left_size == 0 or right_size == 0 or 
                left_size < config.min_samples_leaf or 
                right_size < config.min_samples_leaf):
                continue

            left = dataset_array[left_mask].tolist()
            right = dataset_array[~left_mask].tolist()

            groups = (left, right)
            score = calculate_impurity(groups, class_values, config.criterion)

            if score < best['score']:
                best = {
                    'index': i, 
                    'value': float(split_val), 
                    'score': score, 
                    'groups': groups,
                    'n_samples': len(dataset)
                }

    return best if best['groups'] is not None else None

# TREE BUILDING

def to_terminal(group):
    """Create leaf node with full information"""
    if not group:
        return {
            'type': 'leaf',
            'value': None,
            'samples': 0,
            'distribution': {},
            'impurity': 0.0
        }

    outcomes = [row[-1] for row in group]
    distribution = Counter(outcomes)
    majority = max(distribution, key=distribution.get)

    # Calculate leaf impurity
    total = len(outcomes)
    impurity = 1.0 - sum((count/total)**2 for count in distribution.values())

    return {
        'type': 'leaf',
        'value': majority,
        'samples': len(group),
        'distribution': dict(distribution),
        'impurity': impurity
    }

def split_node_improved(node, config, depth, all_classes):
    """Improved recursive splitting"""
    if not isinstance(node, dict) or 'groups' not in node:
        return

    left, right = node['groups']
    del node['groups']

    # Handle empty children
    if not left or not right:
        combined = left + right
        terminal = to_terminal(combined)
        node.update(terminal)
        node['left'] = node['right'] = None
        return

    # Check max depth
    if config.max_depth and depth >= config.max_depth:
        node['left'] = to_terminal(left)
        node['right'] = to_terminal(right)
        return

    # Check minimum samples
    if len(left) < config.min_samples_split and len(right) < config.min_samples_split:
        node['left'] = to_terminal(left)
        node['right'] = to_terminal(right)
        return

    # Impurity decrease check
    if config.min_impurity_decrease > 0:
        combined = left + right
        parent_impurity = calculate_impurity([combined], all_classes, config.criterion)
        child_impurity = calculate_impurity([left, right], all_classes, config.criterion)
        impurity_decrease = parent_impurity - child_impurity

        if impurity_decrease < config.min_impurity_decrease:
            node['left'] = to_terminal(left)
            node['right'] = to_terminal(right)
            return

    # Process left child
    if len(left) < config.min_samples_split:
        node['left'] = to_terminal(left)
    else:
        node['left'] = get_split_improved(left, config)
        if node['left'] is not None:
            split_node_improved(node['left'], config, depth + 1, all_classes)
        else:
            node['left'] = to_terminal(left)

    # Process right child
    if len(right) < config.min_samples_split:
        node['right'] = to_terminal(right)
    else:
        node['right'] = get_split_improved(right, config)
        if node['right'] is not None:
            split_node_improved(node['right'], config, depth + 1, all_classes)
        else:
            node['right'] = to_terminal(right)


class ImprovedDecisionTree:
    """Improved Decision Tree Classifier"""

    def __init__(self, config):
        self.config = config
        self.tree = None
        self.classes = None
        self.n_features = None
        self.feature_importances_ = None
        self.max_depth_reached = 0
        self.n_leaves = 0
        self.n_nodes = 0

    def fit(self, X, y):
        """Train the decision tree"""
        # Convert to internal format
        dataset = [list(X[i]) + [y[i]] for i in range(len(X))]

        self.classes = sorted(set(y))
        self.n_features = X.shape[1] if hasattr(X, 'shape') else len(X[0])

        # Build tree
        self.tree = get_split_improved(dataset, self.config)
        if self.tree is not None:
            split_node_improved(self.tree, self.config, 1, self.classes)
        else:
            self.tree = to_terminal(dataset)

        # Calculate statistics
        self._calculate_tree_stats()
        self._calculate_feature_importances()

        return self

    def predict(self, X):
        """Predict class labels"""
        return [self._predict_single(row) for row in X]

    def _predict_single(self, row):
        """Predict single instance"""
        node = self.tree

        while isinstance(node, dict):
            if node.get('type') == 'leaf':
                return node.get('value')

            if 'index' in node and len(row) > node['index']:
                node = node['left'] if row[node['index']] < node['value'] else node['right']
            else:
                return node.get('value')

        return node

    def predict_proba(self, X):
        """Predict class probabilities"""
        probas = []
        for row in X:
            leaf = self._get_leaf(row)
            if isinstance(leaf, dict) and 'distribution' in leaf:
                total = leaf['samples']
                if total > 0:
                    probs = [leaf['distribution'].get(c, 0) / total for c in self.classes]
                else:
                    probs = [1.0 / len(self.classes)] * len(self.classes)
            else:
                pred = self._predict_single(row)
                probs = [1.0 if c == pred else 0.0 for c in self.classes]
            probas.append(probs)
        return np.array(probas)

    def _get_leaf(self, row):
        """Navigate to leaf node"""
        node = self.tree

        while isinstance(node, dict) and 'index' in node:
            if len(row) > node['index']:
                node = node['left'] if row[node['index']] < node['value'] else node['right']
            else:
                break

        return node

    def _calculate_tree_stats(self):
        """Calculate tree statistics"""
        def traverse(node, depth=0):
            if not isinstance(node, dict):
                return depth, 1, 1

            if node.get('type') == 'leaf':
                self.n_leaves += 1
                return depth, 1, 1

            self.n_nodes += 1

            left_depth, left_nodes, left_leaves = 0, 0, 0
            right_depth, right_nodes, right_leaves = 0, 0, 0

            if node.get('left'):
                left_depth, left_nodes, left_leaves = traverse(node['left'], depth + 1)
            if node.get('right'):
                right_depth, right_nodes, right_leaves = traverse(node['right'], depth + 1)

            max_depth = max(left_depth, right_depth)
            return max_depth, 1 + left_nodes + right_nodes, left_leaves + right_leaves

        self.n_leaves = 0
        self.n_nodes = 0
        self.max_depth_reached, _, _ = traverse(self.tree)

    def _calculate_feature_importances(self):
        """Calculate feature importance"""
        importance = np.zeros(self.n_features)

        def traverse(node):
            if not isinstance(node, dict) or node.get('type') == 'leaf':
                return

            if 'index' in node:
                feature_idx = node['index']
                n_samples = node.get('n_samples', 0)
                importance[feature_idx] += n_samples

                if node.get('left'):
                    traverse(node['left'])
                if node.get('right'):
                    traverse(node['right'])

        traverse(self.tree)

        if np.sum(importance) > 0:
            self.feature_importances_ = importance / np.sum(importance)
        else:
            self.feature_importances_ = importance

    def get_depth(self):
        """Get tree depth"""
        return self.max_depth_reached

    def get_n_leaves(self):
        """Get number of leaves"""
        return self.n_leaves


# ============================================================================
# RANDOM FOREST IMPLEMENTATION
# ============================================================================

class RandomForestConfig:
    """Configuration for Random Forest"""

    def __init__(self, n_estimators=100, random_seed=42, test_size=0.3,
                 max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 criterion='gini', min_impurity_decrease=0.0,
                 max_features='sqrt', bootstrap=True, 
                 oob_score=False, n_jobs=1, max_samples=None):
        self.n_estimators = n_estimators
        self.random_seed = random_seed
        self.test_size = test_size
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.min_impurity_decrease = min_impurity_decrease
        self.max_features = max_features  # 'sqrt', 'log2', None, or int
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.n_jobs = n_jobs
        self.max_samples = max_samples
        seed(self.random_seed) if self.random_seed else None


class ImprovedRandomForest:
    """Custom Random Forest Classifier using Bootstrap Aggregation"""

    def __init__(self, config):
        self.config = config
        self.trees = []
        self.classes = None
        self.n_features = None
        self.feature_importances_ = None
        self.oob_score_ = None
        self.oob_decision_function_ = None
        self.bootstrap_indices = []

    def fit(self, X, y):
        """Train the random forest"""
        np.random.seed(self.config.random_seed)

        # Store basic info
        self.classes = sorted(set(y))
        self.n_features = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        n_samples = len(X)

        # Determine max_samples
        if self.config.max_samples is None:
            max_samples = n_samples
        elif isinstance(self.config.max_samples, float):
            max_samples = int(self.config.max_samples * n_samples)
        else:
            max_samples = self.config.max_samples

        # Build trees
        self.trees = []
        self.bootstrap_indices = []

        # Sequential training (parallel can be added with ThreadPoolExecutor)
        for i in range(self.config.n_estimators):
            # Create bootstrap sample
            if self.config.bootstrap:
                indices = np.random.choice(n_samples, size=max_samples, replace=True)
                X_bootstrap = X[indices]
                y_bootstrap = y[indices]
                self.bootstrap_indices.append(indices)
            else:
                X_bootstrap = X
                y_bootstrap = y
                self.bootstrap_indices.append(np.arange(n_samples))

            # Create tree config with unique random seed
            tree_config = TreeConfig(
                random_seed=self.config.random_seed + i if self.config.random_seed else None,
                max_depth=self.config.max_depth,
                min_samples_split=self.config.min_samples_split,
                min_samples_leaf=self.config.min_samples_leaf,
                criterion=self.config.criterion,
                min_impurity_decrease=self.config.min_impurity_decrease,
                max_features=self.config.max_features,
                splitter='best'
            )

            # Train tree
            tree = ImprovedDecisionTree(tree_config)
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)

        # Calculate feature importances
        self._calculate_feature_importances()

        # Calculate OOB score if requested
        if self.config.oob_score and self.config.bootstrap:
            self._calculate_oob_score(X, y)

        return self

    def predict(self, X):
        """Predict class labels using majority voting"""
        # Get predictions from all trees
        tree_predictions = np.array([tree.predict(X) for tree in self.trees])

        # Majority voting
        predictions = []
        for i in range(len(X)):
            votes = tree_predictions[:, i]
            prediction = Counter(votes).most_common(1)[0][0]
            predictions.append(prediction)

        return np.array(predictions)

    def predict_proba(self, X):
        """Predict class probabilities by averaging tree probabilities"""
        # Get probability predictions from all trees
        all_probas = np.array([tree.predict_proba(X) for tree in self.trees])

        # Average probabilities across all trees
        avg_probas = np.mean(all_probas, axis=0)

        return avg_probas

    def _calculate_feature_importances(self):
        """Calculate feature importance by averaging across all trees"""
        if not self.trees:
            self.feature_importances_ = np.zeros(self.n_features)
            return

        # Aggregate feature importances from all trees
        importances = np.zeros(self.n_features)

        for tree in self.trees:
            if tree.feature_importances_ is not None:
                importances += tree.feature_importances_

        # Normalize
        if np.sum(importances) > 0:
            self.feature_importances_ = importances / np.sum(importances)
        else:
            self.feature_importances_ = importances

    def _calculate_oob_score(self, X, y):
        """Calculate Out-of-Bag score"""
        n_samples = len(X)
        oob_predictions = np.zeros((n_samples, len(self.classes)))
        oob_counts = np.zeros(n_samples)

        # For each sample, aggregate predictions from trees that didn't see it
        for tree_idx, tree in enumerate(self.trees):
            # Get out-of-bag samples (samples not in bootstrap)
            bootstrap_idx = set(self.bootstrap_indices[tree_idx])
            oob_idx = [i for i in range(n_samples) if i not in bootstrap_idx]

            if len(oob_idx) == 0:
                continue

            # Predict OOB samples
            X_oob = X[oob_idx]
            probas = tree.predict_proba(X_oob)

            # Accumulate predictions
            for i, idx in enumerate(oob_idx):
                oob_predictions[idx] += probas[i]
                oob_counts[idx] += 1

        # Calculate OOB score
        valid_samples = oob_counts > 0
        if np.sum(valid_samples) == 0:
            self.oob_score_ = 0.0
            self.oob_decision_function_ = None
            return

        # Average predictions
        oob_predictions[valid_samples] /= oob_counts[valid_samples, np.newaxis]
        self.oob_decision_function_ = oob_predictions

        # Get final predictions
        oob_pred_labels = np.array([self.classes[np.argmax(pred)] for pred in oob_predictions])

        # Calculate accuracy on valid OOB samples
        y_valid = np.array(y)[valid_samples]
        oob_pred_valid = oob_pred_labels[valid_samples]
        self.oob_score_ = np.mean(y_valid == oob_pred_valid)

    def get_params(self):
        """Get forest parameters"""
        return {
            'n_estimators': self.config.n_estimators,
            'n_trees': len(self.trees),
            'n_features': self.n_features,
            'n_classes': len(self.classes) if self.classes else 0,
            'oob_score': self.oob_score_
        }


# ============================================================================
# EVALUATION METRICS
# ============================================================================

def calculate_metrics(y_true, y_pred, y_proba, classes):
    """Calculate comprehensive metrics"""
    accuracy = accuracy_score(y_true, y_pred)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    y_true_bin = label_binarize(y_true, classes=classes)

    if y_proba is not None and len(classes) > 1:
        try:
            if len(classes) == 2:
                fpr, tpr, _ = roc_curve(y_true_bin, y_proba[:, 1])
                auroc = auc(fpr, tpr)
            else:
                auc_scores = []
                for i in range(len(classes)):
                    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
                    auc_scores.append(auc(fpr, tpr))
                auroc = np.mean(auc_scores)
        except:
            auroc = 0.0
    else:
        auroc = 0.0

    return {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'auroc': auroc
    }


# ============================================================================
# COMPARISON FUNCTIONS
# ============================================================================

def print_rf_comparison_table(sk_metrics, custom_metrics, sk_oob=None, custom_oob=None):
    """Print comparison table for Random Forest"""
    print("\n┌──────────────┬─────────────┬─────────────┐")
    print("│ Metric       │ sklearn     │ Custom      │")
    print("├──────────────┼─────────────┼─────────────┤")
    print(f"│ Accuracy     │ {sk_metrics['accuracy']*100:>6.2f}%     │ {custom_metrics['accuracy']*100:>6.2f}%     │")
    print(f"│ F1-Score     │ {sk_metrics['f1_weighted']:>7.4f}     │ {custom_metrics['f1_weighted']:>7.4f}     │")
    print(f"│ AUROC        │ {sk_metrics['auroc']:>7.4f}     │ {custom_metrics['auroc']:>7.4f}     │")

    if sk_oob is not None and custom_oob is not None:
        print(f"│ OOB Score    │ {sk_oob*100:>6.2f}%     │ {custom_oob*100:>6.2f}%     │")

    print("└──────────────┴─────────────┴─────────────┘")


def run_rf_comparison(dataset_loader, dataset_name, config_name,config):
    """Compare custom Random Forest with sklearn"""

    print("=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    # Load data
    data = dataset_loader()
    X, y = data.data, data.target
    classes = sorted(set(y))

    print(f"\nDataset Info:")
    print(f"  Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(classes)}")

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.test_size,
        random_state=config.random_seed, stratify=y
    )

    # sklearn Random Forest
    rf_sklearn = RandomForestClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        min_samples_split=config.min_samples_split,
        min_samples_leaf=config.min_samples_leaf,
        criterion=config.criterion,
        max_features=config.max_features,
        bootstrap=config.bootstrap,
        oob_score=config.oob_score and config.bootstrap,
        random_state=config.random_seed,
        n_jobs=-1
    )
    rf_sklearn.fit(X_train, y_train)
    y_pred_sk = rf_sklearn.predict(X_test)
    y_proba_sk = rf_sklearn.predict_proba(X_test)

    metrics_sk = calculate_metrics(y_test, y_pred_sk, y_proba_sk, classes)
    sk_oob = rf_sklearn.oob_score_ if config.oob_score and config.bootstrap else None

    # Custom Random Forest
    rf_custom = ImprovedRandomForest(config)
    rf_custom.fit(X_train, y_train)
    y_pred_custom = rf_custom.predict(X_test)
    y_proba_custom = rf_custom.predict_proba(X_test)

    metrics_custom = calculate_metrics(y_test, y_pred_custom, y_proba_custom, classes)
    custom_oob = rf_custom.oob_score_ if config.oob_score and config.bootstrap else None

    # Print comparison
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    print_rf_comparison_table(metrics_sk, metrics_custom, sk_oob, custom_oob)

    # # Print feature importances comparison
    # print("\nTop 5 Feature Importances:")
    # print("  sklearn:")
    # sk_top = sorted(enumerate(rf_sklearn.feature_importances_), 
    #                key=lambda x: x[1], reverse=True)[:5]
    # for idx, imp in sk_top:
    #     print(f"    Feature {idx}: {imp:.4f}")

    # print("  Custom:")
    # custom_top = sorted(enumerate(rf_custom.feature_importances_), 
    #                    key=lambda x: x[1], reverse=True)[:5]
    # for idx, imp in custom_top:
    #     print(f"    Feature {idx}: {imp:.4f}")

    return {
        'dataset': dataset_name,
        'config': config_name,
        'sklearn_accuracy': metrics_sk['accuracy'],
        'sklearn_f1': metrics_sk['f1_weighted'],
        'sklearn_auroc': metrics_sk['auroc'],
        'sklearn_oob': sk_oob if sk_oob is not None else 0.0,
        'custom_accuracy': metrics_custom['accuracy'],
        'custom_f1': metrics_custom['f1_weighted'],
        'custom_auroc': metrics_custom['auroc'],
        'custom_oob': custom_oob if custom_oob is not None else 0.0
    }


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 80)
    print(" " * 25 + "RANDOM FOREST ANALYSIS")
    print("=" * 80)

    # Test configurations
    rf_configs = [
        ("RF-100 Trees (Gini, sqrt)", RandomForestConfig(
            n_estimators=100, max_depth=10, criterion='gini', 
            max_features='sqrt', bootstrap=True, oob_score=True
        )),
        ("RF-50 Trees (Entropy, sqrt)", RandomForestConfig(
            n_estimators=50, max_depth=10, criterion='entropy',
            max_features='sqrt', bootstrap=True, oob_score=True
        )),
        ("RF-100 Trees (Gini, log2)", RandomForestConfig(
            n_estimators=100, max_depth=10, criterion='gini',
            max_features='log2', bootstrap=True, oob_score=True
        )),
    ]

    datasets = [
        (load_iris, "IRIS"),
        (load_wine, "WINE")
    ]
    results = []
    for dataset_loader, dataset_name in datasets:
        for config_name, config in rf_configs:
            print(f"\n\n{'='*80}")
            print(f"Configuration: {config_name}")
            print(f"{'='*80}")
            result = run_rf_comparison(dataset_loader, dataset_name, config_name,config)
            results.append(result)

    csv_file = 'random_forest_results.csv'
    fieldnames = ['dataset', 'config', 
                  'sklearn_accuracy', 'sklearn_f1', 'sklearn_auroc', 'sklearn_oob',
                  'custom_accuracy', 'custom_f1', 'custom_auroc', 'custom_oob']

    with open(csv_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print(f'\n\nResults saved to {csv_file}')


                         RANDOM FOREST ANALYSIS


Configuration: RF-100 Trees (Gini, sqrt)
DATASET: IRIS

Dataset Info:
  Samples: 150, Features: 4, Classes: 3

RESULTS

┌──────────────┬─────────────┬─────────────┐
│ Metric       │ sklearn     │ Custom      │
├──────────────┼─────────────┼─────────────┤
│ Accuracy     │  88.89%     │  88.89%     │
│ F1-Score     │  0.8878     │  0.8878     │
│ AUROC        │  0.9889     │  0.9941     │
│ OOB Score    │  95.24%     │  95.24%     │
└──────────────┴─────────────┴─────────────┘


Configuration: RF-50 Trees (Entropy, sqrt)
DATASET: IRIS

Dataset Info:
  Samples: 150, Features: 4, Classes: 3

RESULTS

┌──────────────┬─────────────┬─────────────┐
│ Metric       │ sklearn     │ Custom      │
├──────────────┼─────────────┼─────────────┤
│ Accuracy     │  88.89%     │  88.89%     │
│ F1-Score     │  0.8878     │  0.8878     │
│ AUROC        │  0.9859     │  0.9933     │
│ OOB Score    │  95.24%     │  96.19%     │
└──────────────┴─────────────┴

In [89]:
import numpy as np
from collections import Counter
from sklearn.datasets import load_iris, load_breast_cancer, load_wine, load_digits, make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
                              classification_report, confusion_matrix)
from sklearn.preprocessing import label_binarize
import time

# ============================================================================
# EXTRA TREES IMPLEMENTATION FROM SCRATCH
# ============================================================================

class ExtraDecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, max_features='sqrt'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
    
    def fit(self, X, y):
        self.n_classes = len(set(y))
        self.n_features = X.shape[1]
        
        if self.max_features == 'sqrt':
            self.n_features_split = max(1, int(np.sqrt(self.n_features)))
        elif self.max_features == 'log2':
            self.n_features_split = max(1, int(np.log2(self.n_features)))
        elif isinstance(self.max_features, int):
            self.n_features_split = min(self.max_features, self.n_features)
        elif isinstance(self.max_features, float):
            self.n_features_split = max(1, int(self.max_features * self.n_features))
        else:
            self.n_features_split = self.n_features
        
        self.tree = self._grow_tree(X, y)
    
    def predict(self, X):
        return np.array([self._predict(inputs) for inputs in X])
    
    def predict_proba(self, X):
        return np.array([self._predict_proba(inputs) for inputs in X])
    
    def _predict(self, inputs):
        node = self.tree
        while not node.is_leaf_node():
            if inputs[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value
    
    def _predict_proba(self, inputs):
        node = self.tree
        while not node.is_leaf_node():
            if inputs[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.proba
    
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_labels == 1 or \
           n_samples < self.min_samples_split:
            leaf_value = self._most_common_label(y)
            leaf_proba = self._class_probabilities(y)
            return ExtraNode(value=leaf_value, proba=leaf_proba)
        
        feat_idxs = np.random.choice(n_features, self.n_features_split, replace=False)
        best_feat, best_thresh = self._best_random_split(X, y, feat_idxs)
        
        if best_feat is None:
            leaf_value = self._most_common_label(y)
            leaf_proba = self._class_probabilities(y)
            return ExtraNode(value=leaf_value, proba=leaf_proba)
        
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return ExtraNode(best_feat, best_thresh, left, right)
    
    def _best_random_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None
        
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            min_val, max_val = X_column.min(), X_column.max()
            
            if min_val == max_val:
                continue
            
            threshold = np.random.uniform(min_val, max_val)
            gain = self._information_gain(y, X_column, threshold)
            
            if gain > best_gain:
                best_gain = gain
                split_idx = feat_idx
                split_thresh = threshold
        
        return split_idx, split_thresh
    
    def _information_gain(self, y, X_column, split_thresh):
        parent_entropy = self._entropy(y)
        left_idxs, right_idxs = self._split(X_column, split_thresh)
        
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
        
        n_left, n_right = len(left_idxs), len(right_idxs)
        n_total = len(y)
        e_left = self._entropy(y[left_idxs])
        e_right = self._entropy(y[right_idxs])
        child_entropy = (n_left / n_total) * e_left + (n_right / n_total) * e_right
        
        return parent_entropy - child_entropy
    
    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs
    
    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])
    
    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]
    
    def _class_probabilities(self, y):
        counter = Counter(y)
        total = len(y)
        proba = np.zeros(self.n_classes)
        for class_label, count in counter.items():
            proba[class_label] = count / total
        return proba


class ExtraNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, 
                 *, value=None, proba=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.proba = proba
    
    def is_leaf_node(self):
        return self.value is not None


class ExtraTreesClassifierCustom:
    """
    Extra Trees Classifier with Configurable Hyperparameters
    
    Parameters:
    - n_estimators: Number of trees
    - max_depth: Maximum tree depth
    - min_samples_split: Minimum samples to split a node
    - max_features: Features per split ('sqrt', 'log2', int, float, None)
    - random_state: Random seed
    """
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, 
                 max_features='sqrt', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.trees = []
        
        for i in range(self.n_estimators):
            tree = ExtraDecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )
            tree.fit(X, y)
            self.trees.append(tree)
        
        return self
    
    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        y_pred = [Counter(tree_pred).most_common(1)[0][0] for tree_pred in tree_preds]
        return np.array(y_pred)
    
    def predict_proba(self, X):
        tree_probas = np.array([tree.predict_proba(X) for tree in self.trees])
        avg_proba = np.mean(tree_probas, axis=0)
        return avg_proba


# ============================================================================
# COMPARISON FUNCTION
# ============================================================================

def compare_models(X_train, X_test, y_train, y_test, dataset_name, config):
    """Compare Custom vs Sklearn Extra Trees"""
    
    print("=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)
    # print(f"Config: n_trees={config['n_estimators']}, depth={config['max_depth']}, "
    #       f"min_split={config['min_samples_split']}, features={config['max_features']}")
    
    n_classes = len(np.unique(y_train))
    
    # Custom Extra Trees
    
    custom_model = ExtraTreesClassifierCustom(**config)
    
    start = time.time()
    custom_model.fit(X_train, y_train)
    custom_train_time = time.time() - start
    
    start = time.time()
    custom_pred = custom_model.predict(X_test)
    custom_predict_time = time.time() - start
    
    custom_proba = custom_model.predict_proba(X_test)
    
    custom_acc = accuracy_score(y_test, custom_pred)
    custom_f1 = f1_score(y_test, custom_pred, average='macro' if n_classes > 2 else 'binary')
    
    if n_classes == 2:
        custom_auroc = roc_auc_score(y_test, custom_proba[:, 1])
    else:
        y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
        custom_auroc = roc_auc_score(y_test_bin, custom_proba, average='macro', multi_class='ovr')
    # Sklearn Extra Trees
    sklearn_model = ExtraTreesClassifier(**config)
    
    start = time.time()
    sklearn_model.fit(X_train, y_train)
    sklearn_train_time = time.time() - start
    
    start = time.time()
    sklearn_pred = sklearn_model.predict(X_test)
    sklearn_predict_time = time.time() - start
    
    sklearn_proba = sklearn_model.predict_proba(X_test)
    
    sklearn_acc = accuracy_score(y_test, sklearn_pred)
    sklearn_f1 = f1_score(y_test, sklearn_pred, average='macro' if n_classes > 2 else 'binary')
    
    if n_classes == 2:
        sklearn_auroc = roc_auc_score(y_test, sklearn_proba[:, 1])
    else:
        y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
        sklearn_auroc = roc_auc_score(y_test_bin, sklearn_proba, average='macro', multi_class='ovr')
    # Comparison
    print(f"\n{'─'*90}")
    print("COMPARISON")
    print(f"{'─'*90}")
    print(f"{'Metric':<25} {'Custom':<15} {'Sklearn':<15}")
    print(f"{'─'*90}")
    print(f"{'Accuracy':<25} {custom_acc:<15.4f} {sklearn_acc:<15.4f} ")
    print(f"{'F1-Score':<25} {custom_f1:<15.4f} {sklearn_f1:<15.4f}  ")
    print(f"{'AUROC':<25} {custom_auroc:<15.4f} {sklearn_auroc:<15.4f} ")
    
    return {
        'dataset': dataset_name,
        'custom': {'acc': custom_acc, 'f1': custom_f1, 'auroc': custom_auroc, 'train_time': custom_train_time},
        'sklearn': {'acc': sklearn_acc, 'f1': sklearn_f1, 'auroc': sklearn_auroc, 'train_time': sklearn_train_time}
    }


# ============================================================================
# MAIN: TEST ON ALL DATASET TYPES
# ============================================================================
if __name__ == "__main__":
    print("\n" + "=" * 80)
    print(" " * 25 + "EXTRA TREE ANALYSIS")
    print("=" * 80)
    
    results = []
    config = {
        'n_estimators': 100,
        'max_depth': 15,
        'min_samples_split': 5,
        'max_features': 'sqrt',
        'random_state': 42
    }
    
    # Dataset 1: Iris
    data = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        data.data, data.target, test_size=0.2, random_state=42
    )
    result1 = compare_models(X_train, X_test, y_train, y_test, "Iris", config)
    results.append(result1)
    
    # Dataset 2: Wine
    data = load_wine()
    X_train, X_test, y_train, y_test = train_test_split(
        data.data, data.target, test_size=0.2, random_state=42
    )
    result2 = compare_models(X_train, X_test, y_train, y_test, "Wine", config)
    results.append(result2)
    
    # Prepare data for CSV (flatten the nested dictionary)
    csv_data = []
    for result in results:
        csv_data.append({
            'dataset': result['dataset'],
            'config': f"n_est={config['n_estimators']}, depth={config['max_depth']}, features={config['max_features']}",
            'sklearn_accuracy': result['sklearn']['acc'],
            'sklearn_f1': result['sklearn']['f1'],
            'sklearn_auroc': result['sklearn']['auroc'],
            'sklearn_train_time': result['sklearn']['train_time'],
            'custom_accuracy': result['custom']['acc'],
            'custom_f1': result['custom']['f1'],
            'custom_auroc': result['custom']['auroc'],
            'custom_train_time': result['custom']['train_time']
        })
    
    # Save to CSV
    csv_file = 'extra_tree_results.csv'
    fieldnames = ['dataset', 'config', 
                  'sklearn_accuracy', 'sklearn_f1', 'sklearn_auroc', 'sklearn_train_time',
                  'custom_accuracy', 'custom_f1', 'custom_auroc', 'custom_train_time']
    
    with open(csv_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_data)
    
    print(f'\n\nResults saved to {csv_file}')



                         EXTRA TREE ANALYSIS
DATASET: Iris

──────────────────────────────────────────────────────────────────────────────────────────
COMPARISON
──────────────────────────────────────────────────────────────────────────────────────────
Metric                    Custom          Sklearn        
──────────────────────────────────────────────────────────────────────────────────────────
Accuracy                  1.0000          1.0000          
F1-Score                  1.0000          1.0000           
AUROC                     1.0000          1.0000          
DATASET: Wine

──────────────────────────────────────────────────────────────────────────────────────────
COMPARISON
──────────────────────────────────────────────────────────────────────────────────────────
Metric                    Custom          Sklearn        
──────────────────────────────────────────────────────────────────────────────────────────
Accuracy                  1.0000          1.0000          
F1-